# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: fetch latest code</h2>
            <span style="color:#f71;">I'm continually improving these labs, adding more examples and exercises.
            At the start of each week, it's worth checking you have the latest code.<br/>
            First do a <a href="https://chatgpt.com/share/6734e705-3270-8012-a074-421661af6ba9">git pull and merge your changes as needed</a>. Any problems? Try asking ChatGPT to clarify how to merge - or contact me!<br/><br/>
            After you've pulled the code, from the llm_engineering directory, in a Cursor Terminal, run:<br/>
            <code>uv sync</code><br/>
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use high end models GPT 5, Claude 4.5 Sonnet, Gemini 2.5 Pro, Grok 4, which are the slightly higher priced models. The costs are still low, but if you'd prefer to keep costs ultra low, please pick lower cost models like gpt-5-nano.
            </span>
        </td>
    </tr>
</table>

In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [2]:
# Connect to OpenAI, Anthropic and Google; comment out the Claude or Google lines if you're not using them

# openai = OpenAI()
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

minimax = "minimax-m3:cloud";
llama = "llama3.2:3b"
phi3 = "phi3"

In [3]:
# Connect to OpenAI, Anthropic and Google; comment out the Claude or Google lines if you're not using them

# openai = OpenAI()
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

minimax = "minimax-m3:cloud";
llama = "llama3.2:3b"
phi3 = "phi3"


## PLEASE NOTE:

We will be writing a solution to convert Python into efficient, optimized C++ code for your machine, which can be compiled to native machine code and executed.

It is not necessary for you to execute the code yourself - that's not the point of the exercise!

But if you would like to (because it's satisfying!) then I'm including the steps here. Very optional!

As an alternative, I'll also show you a website where you can run the C++ code.

In [4]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.26200',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': ''},
 'package_managers': ['winget'],
 'cpu': {'brand': 'Intel(R) Core(TM) i5-9600 CPU @ 3.10GHz',
  'cores_logical': 6,
  'cores_physical': 6,
  'simd': []},
 'toolchain': {'compilers': {'gcc': '', 'g++': '', 'clang': '', 'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [5]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = ollama.chat.completions.create(model=minimax, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))
    

Yes — you need to install a C++ compiler. Your system report shows all compiler fields are empty (`gcc`, `g++`, `clang`, `msvc_cl` are all blank), so no C++ toolchain is currently available.

## Simplest install: MinGW-w64 (provides `g++`)

The fastest way to get up and running is a single `winget` command. WinLibs is a standalone MinGW-w64 distribution that ships `g++` and automatically adds it to your `PATH`.

1. Open PowerShell or Windows Terminal.
2. Run:
   ```powershell
   winget install -e --id BrechtSanders.WinLibs.POSIX.UCRT
   ```
3. Accept any prompts (admin elevation, license agreement).
4. **Close and reopen your terminal** (and any IDE/editor) so the new `PATH` is picked up.
5. Verify it worked:
   ```powershell
   g++ --version
   ```
   You should see a GCC version (e.g., `g++ (GCC) 13.x.x` or similar).

That's it — no Visual Studio installer, no manual PATH editing.

## Python code to compile and run `main.cpp`

For the fastest practical runtime performance on your Intel i5-9600 (Coffee Lake), use `-O3 -march=native` and a modern C++ standard:

```python
import subprocess

compile_command = [
    "g++",
    "-O3",
    "-march=native",
    "-std=c++17",   # bump to c++20 / c++23 if you want
    "main.cpp",
    "-o", "main.exe",
]
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)

run_command = ["main.exe"]   # equivalent to ./main.exe on Windows
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)

print(run_result.stdout)
```

### Notes
- **`-march=native`** tunes the generated code specifically for your CPU, which gives the biggest single performance win on a single-file build (and is safe here because you're compiling and running on the same machine).
- **`-O3`** enables the highest standard optimization level.
- If you want even more speed and your code does heavy floating-point math, you can add `-ffast-math`, but be aware it relaxes IEEE 754 compliance and may change numerical results.
- For `-std=`, `c++17` is the safe default; use `c++20` or `c++23` if your code uses newer features.
- `subprocess.run` is invoked with a list, so no shell quoting issues — works regardless of whether `main.cpp` has spaces in the path.

## If you need to install something

If you would like to, please follow GPTs instructions! Then rerun the analysis afterwards (you might need to Restart the notebook) to confirm you're set.

You should now be equipped with the command to compile the code, and the command to run it!

Enter that in the cell below:

In [7]:
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

## And now, on with the main task

In [8]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [9]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [10]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [11]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [12]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [13]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [14]:
run_python(pi)

Result: 3.141592656089
Execution Time: 33.061703 seconds


In [15]:
port(ollama, minimax, pi)

# Compiling C++ and executing

This next cell contains the command to compile a C++ file based on the instructions from GPT.

Again, it's not crucial to do this step if you don't wish to!

OR alternatively: student Sandeep K.G. points out that you can run Python and C++ code online to test it out that way. Thank you Sandeep!  
> Not an exact comparison but you can still get the idea of performance difference.  
> For example here: https://www.programiz.com/cpp-programming/online-compiler/

In [16]:
# Use the commands from GPT 5

def compile_and_run():
    subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)

In [ ]:
compile_and_run()

In [18]:
19.178207/0.082168

233.40238292279233

## OK let's try the other contenders!

In [ ]:
port(ollama, phi3, pi)
compile_and_run()

In [ ]:
port(ollama, llama, pi)
compile_and_run()

In [31]:
print(f"""
In Ed's experiments, the performance speedups were:

4th place: Claude Sonnet 4.5: {19.178207/0.104241:.0f}X speedup
3rd place: GPT-5: {19.178207/0.082168:.0f}X speedup
2nd place: Grok 4: {19.178207/0.018092:.0f}X speedup
1st place: Gemini 2.5 Pro: {19.178207/0.013314:.0f}X speedup
""")


In Ed's experiments, the performance speedups were:

4th place: Claude Sonnet 4.5: 184X speedup
3rd place: GPT-5: 233X speedup
2nd place: Grok 4: 1060X speedup
1st place: Gemini 2.5 Pro: 1440X speedup

